# Initial setup
The below code has to be run only once to download all necessary files and convert them to the necessary formats used in other notebooks in this directory.

In [1]:
import torch

#device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [2]:
# downloading the model
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    local_dir="./model/biomedclip",
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

'/elias/data/home/University/InformatikAI/26W/AI and Visualization/UE/project/git/subgroup_1/model/biomedclip'

In [3]:
# downloading the BoghdadyJR/radiology dataset
from datasets import load_dataset

ds = load_dataset("BoghdadyJR/radiology", split="train")
ds.save_to_disk("./datasets/radiology/train")
ds = load_dataset("BoghdadyJR/radiology", split="test")
ds.save_to_disk("./datasets/radiology/test")

Saving the dataset (0/12 shards):   0%|          | 0/28551 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7124 [00:00<?, ? examples/s]

In [9]:
# automatically download ROCOv2 dataset from Zenodo and save to disk
from pathlib import Path
from urllib.request import urlretrieve

def download(path: str, url: str):
    output_path = Path(path)
    if output_path.exists():
        print(f"{output_path} already exists, skipping download.")
        return
    output_path.parent.mkdir(parents=True, exist_ok=True)
    progress = tqdm(unit="B", unit_scale=True, desc=output_path.name)
    def reporthook(block_num, block_size, total_size):
        if total_size > 0:
            progress.total = total_size
        downloaded = block_num * block_size
        progress.update(max(0, downloaded - progress.n))
    try:
        urlretrieve(url, output_path, reporthook=reporthook)
    finally:
        progress.close()

download("./datasets/ROCOv2/cui_mapping.csv", "https://zenodo.org/records/10821435/files/cui_mapping.csv?download=1")
download("./datasets/ROCOv2/license_information.csv", "https://zenodo.org/records/10821435/files/license_information.csv?download=1")
download("./datasets/ROCOv2/test_captions.csv", "https://zenodo.org/records/10821435/files/test_captions.csv?download=1")
download("./datasets/ROCOv2/test_concepts.csv", "https://zenodo.org/records/10821435/files/test_concepts.csv?download=1")
download("./datasets/ROCOv2/test_concepts_manual.csv", "https://zenodo.org/records/10821435/files/test_concepts_manual.csv?download=1")
download("./datasets/ROCOv2/test_images.zip", "https://zenodo.org/records/10821435/files/test_images.zip?download=1")
download("./datasets/ROCOv2/train_captions.csv", "https://zenodo.org/records/10821435/files/train_captions.csv?download=1")
download("./datasets/ROCOv2/train_concepts.csv", "https://zenodo.org/records/10821435/files/train_concepts.csv?download=1")
download("./datasets/ROCOv2/train_concepts_manual.csv", "https://zenodo.org/records/10821435/files/train_concepts_manual.csv?download=1")
download("./datasets/ROCOv2/train_images.zip", "https://zenodo.org/records/10821435/files/train_images.zip?download=1")
download("./datasets/ROCOv2/valid_captions.csv", "https://zenodo.org/records/10821435/files/valid_captions.csv?download=1")
download("./datasets/ROCOv2/valid_concepts.csv", "https://zenodo.org/records/10821435/files/valid_concepts.csv?download=1")
download("./datasets/ROCOv2/valid_concepts_manual.csv", "https://zenodo.org/records/10821435/files/valid_concepts_manual.csv?download=1")
download("./datasets/ROCOv2/valid_images.zip", "https://zenodo.org/records/10821435/files/valid_images.zip?download=1")

datasets/ROCOv2/cui_mapping.csv already exists, skipping download.
datasets/ROCOv2/license_information.csv already exists, skipping download.
datasets/ROCOv2/test_captions.csv already exists, skipping download.
datasets/ROCOv2/test_concepts.csv already exists, skipping download.
datasets/ROCOv2/test_concepts_manual.csv already exists, skipping download.
datasets/ROCOv2/test_images.zip already exists, skipping download.
datasets/ROCOv2/train_captions.csv already exists, skipping download.
datasets/ROCOv2/train_concepts.csv already exists, skipping download.
datasets/ROCOv2/train_concepts_manual.csv already exists, skipping download.
datasets/ROCOv2/train_images.zip already exists, skipping download.
datasets/ROCOv2/valid_captions.csv already exists, skipping download.
datasets/ROCOv2/valid_concepts.csv already exists, skipping download.
datasets/ROCOv2/valid_concepts_manual.csv already exists, skipping download.
datasets/ROCOv2/valid_images.zip already exists, skipping download.


In [10]:
# loading ROCOv2 dataset from raw files first
import csv
import os
import zipfile
from datasets import Dataset, Features, Sequence, Value, Image as HFImage
from tqdm import tqdm
from pathlib import Path
import pickle as pkl

def convert_concepts(concept_string: str, cui_to_idx: dict[str, int]) -> list[int]:
    if not concept_string:
        return []
    return [cui_to_idx[cui] for cui in concept_string.split(";") if cui and cui in cui_to_idx]

def load_roco_dataset(
    path: str,
    split: str
) -> tuple[Dataset, list[str]]:
    # check if all files exist
    required_files = [
        f"{split}_captions.csv",
        f"{split}_concepts_manual.csv",
        f"{split}_concepts.csv",
        f"{split}_images.zip",
        "cui_mapping.csv",
    ]
    for filename in required_files:
        filepath = os.path.join(path, filename)
        if not os.path.isfile(filepath):
            raise FileNotFoundError(
                f"Required file not found: {filepath}. "
                f"The split may be unknown or incomplete."
            )
    # load cui mapping
    cui_to_idx = {}
    concept_mapping = []

    with open(os.path.join(path, "cui_mapping.csv"), "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        _ = next(reader) # header line
        for row in reader:
            cui = row[0]
            concept_name = row[1]
            idx = len(concept_mapping)
            cui_to_idx[cui] = idx
            concept_mapping.append(concept_name)

    with open(os.path.join(path, f"{split}_captions.csv"), "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        _ = next(reader) # header line
        caption_dict = {
            row[0]: row[1]
            for row in list(reader)
        }

    with open(os.path.join(path, f"{split}_concepts.csv"), "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        _ = next(reader) # header line
        concepts_dict = {
            row[0]: row[1]
            for row in list(reader)
        }

    with open(os.path.join(path, f"{split}_concepts_manual.csv"), "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        _ = next(reader) # header line
        concepts_manual_dict = {
            row[0]: row[1]
            for row in list(reader)
        }
    
    features = Features({
        "id": Value("string"),
        "image": HFImage(),
        "concepts": Sequence(Value("int64")),
        "concepts_manual": Sequence(Value("int64")),
        "caption": Value("string"),
    })

    def generate_samples():
        with zipfile.ZipFile(os.path.join(path, f"{split}_images.zip"), "r") as archive:
            image_files = {
                Path(name).name: name
                for name in archive.namelist()
                if name.lower().endswith(".jpg")
            }

            for image_id, caption in tqdm(caption_dict.items(), desc="Processing images"):
                if image_id not in concepts_dict:
                    raise ValueError(f"Missing concepts entry for ID {image_id}")
                if image_id not in concepts_manual_dict:
                    raise ValueError(f"Missing manual concepts entry for ID {image_id}")
                image_filename = f"{image_id}.jpg"
                if image_filename not in image_files:
                    raise FileNotFoundError(f"Image {image_filename} not found in archive.")
                image_bytes = archive.read(image_files[image_filename])
                yield {
                    "id": image_id,
                    "image": {
                        "bytes": image_bytes,
                        "path": None,
                    },
                    "concepts": convert_concepts(concepts_dict[image_id], cui_to_idx),
                    "concepts_manual": convert_concepts(concepts_manual_dict[image_id], cui_to_idx),
                    "caption": caption,
                }
    dataset = Dataset.from_generator(
        generate_samples,
        features=features,
        writer_batch_size=16,
        keep_in_memory=False,
    )
    return dataset, concept_mapping

ds, cui = load_roco_dataset("./datasets/ROCOv2", "test")
ds.save_to_disk("./datasets/roco/test")
with open("./datasets/roco/test/cui.pkl", "wb") as f:
    pkl.dump(cui, f)
ds, cui = load_roco_dataset("./datasets/ROCOv2", "train")
ds.save_to_disk("./datasets/roco/train")
with open("./datasets/roco/train/cui.pkl", "wb") as f:
    pkl.dump(cui, f)
    ds, cui = load_roco_dataset("./datasets/ROCOv2", "valid")
ds.save_to_disk("./datasets/roco/valid")
with open("./datasets/roco/valid/cui.pkl", "wb") as f:
    pkl.dump(cui, f)

Generating train split: 0 examples [00:00, ? examples/s]

Processing images: 100%|██████████| 9927/9927 [00:04<00:00, 2154.60it/s]


Saving the dataset (0/2 shards):   0%|          | 0/9927 [00:00<?, ? examples/s]

Saving the dataset (0/11 shards):   0%|          | 0/59958 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/9904 [00:00<?, ? examples/s]

In [11]:
# reloading the dataset BoghdadyJR/radiology from local backup (now includes caching index)
from datasets import load_from_disk
ds_rad = load_from_disk("./datasets/radiology/test")

In [12]:
# reloading the dataset ROCOv2 from local backup (now includes caching index)
from datasets import load_from_disk
import pickle as pkl
ds_roco = load_from_disk("./datasets/roco/test")
with open("./datasets/roco/test/cui.pkl", "rb") as f:
    ds_roco_cui: list[str] = pkl.load(f)

In [13]:
# loading the model from local backup
from open_clip import create_model_and_transforms, get_tokenizer
from open_clip.factory import HF_HUB_PREFIX, _MODEL_CONFIGS
from open_clip.tokenizer import HFTokenizer
from open_clip.model import CLIP
import json
from torchvision.transforms.transforms import Compose
from typing import cast, Callable
from PIL.Image import Image

model_name = "biomedclip_local"
with open("model/biomedclip/open_clip_config.json", "r") as f:
    config = json.load(f)
    model_cfg = config["model_cfg"]
    preprocess_cfg = config["preprocess_cfg"]


if (not model_name.startswith(HF_HUB_PREFIX)
    and model_name not in _MODEL_CONFIGS
    and config is not None):
    _MODEL_CONFIGS[model_name] = model_cfg

tokenizer: HFTokenizer = cast(
    HFTokenizer,
    get_tokenizer("local-dir:model/biomedclip"),
)

model, _, preprocess = cast(
    tuple[CLIP, None, Callable[[Image], torch.Tensor]],
    create_model_and_transforms(
    model_name=model_name,
    pretrained="model/biomedclip/open_clip_pytorch_model.bin",
    **{f"image_{k}": v for k, v in preprocess_cfg.items()},
))
model.to(device)

if not isinstance(preprocess, Compose):
    raise ValueError(f"Expected Compose, got {type(preprocess)}")
if not isinstance(tokenizer, HFTokenizer):
    raise ValueError(f"Expected HFTokenizer, got {type(tokenizer)}")

model.eval()
None

In [ ]:
# run embeddings for all the data and save them to disk
from PIL.Image import Image
import pickle as pkl
from tqdm import tqdm
context_length = 256
batch_size = 100
dataset = ds_roco
image_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
caption_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
concept_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
caption_concept_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
logit_scales = torch.zeros((len(dataset), 1), dtype=torch.float32)

for start in tqdm(range(0, len(dataset), batch_size)):
    end = min(start + batch_size, len(dataset))
    images = cast(list[Image], dataset[start:end]['image'])
    captions = cast(list[str], dataset[start:end]["caption"])
    concepts = cast(list[str], [" ".join([ds_roco_cui[c] for c in cs]) for cs in dataset[start:end]["concepts"]])
    caption_concepts = [f"{c} {cap}" for c, cap in zip(concepts, captions)]
    images_processed = torch.stack([preprocess(img) for img in images]).to(device)
    texts = tokenizer(captions + concepts + caption_concepts, context_length=context_length).to(device)
    with torch.no_grad():
        image_features, text_features, logit_scale = model(images_processed, texts)
        image_embeddings[start:end] = image_features[:len(images)]
        caption_embeddings[start:end] = text_features[:len(captions)]
        concept_embeddings[start:end] = text_features[len(captions):len(captions)+len(concepts)]
        caption_concept_embeddings[start:end] = text_features[len(captions)+len(concepts):]
        logit_scales[start:end] = logit_scale

with open("./datasets/roco/test/image_embeddings.pkl", "wb") as f:
    pkl.dump(image_embeddings.cpu().detach(), f)
with open("./datasets/roco/test/caption_embeddings.pkl", "wb") as f:
    pkl.dump(caption_embeddings.cpu().detach(), f)
with open("./datasets/roco/test/concept_embeddings.pkl", "wb") as f:
    pkl.dump(concept_embeddings.cpu().detach(), f)
with open("./datasets/roco/test/caption_concept_embeddings.pkl", "wb") as f:
    pkl.dump(caption_concept_embeddings.cpu().detach(), f)
with open("./datasets/roco/test/logit_scales.pkl", "wb") as f:
    pkl.dump(logit_scales.cpu().detach(), f)

dataset = ds_rad
image_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
caption_embeddings = torch.zeros((len(dataset), 512), dtype=torch.float32)
logit_scales = torch.zeros((len(dataset), 1), dtype=torch.float32)

for start in tqdm(range(0, len(dataset), batch_size)):
    end = min(start + batch_size, len(dataset))
    images = cast(list[Image], dataset[start:end]['image'])
    captions = cast(list[str], dataset[start:end]["caption"])
    images_processed = torch.stack([preprocess(img) for img in images]).to(device)
    texts = tokenizer(captions, context_length=context_length).to(device)
    with torch.no_grad():
        image_features, text_features, logit_scale = model(images_processed, texts)
        image_embeddings[start:end] = image_features[:len(images)]
        caption_embeddings[start:end] = text_features[:len(captions)]
        logit_scales[start:end] = logit_scale

with open("./datasets/radiology/test/image_embeddings.pkl", "wb") as f:
    pkl.dump(image_embeddings.cpu().detach(), f)
with open("./datasets/radiology/test/caption_embeddings.pkl", "wb") as f:
    pkl.dump(caption_embeddings.cpu().detach(), f)
with open("./datasets/radiology/test/logit_scales.pkl", "wb") as f:
    pkl.dump(logit_scales.cpu().detach(), f)

cui_embeddings = torch.zeros((len(ds_roco_cui), 512), dtype=torch.float32)
images_processed = torch.zeros((0, 3, 224, 224)).to(device)
for start in tqdm(range(0, len(ds_roco_cui), batch_size), desc="Processing concept embeddings"):
    end = min(start + batch_size, len(ds_roco_cui))
    texts = tokenizer(ds_roco_cui[start:end], context_length=context_length).to(device)
    with torch.no_grad():
        _, text_features, _ = model(images_processed, texts)
        cui_embeddings[start:end] = text_features.cpu().detach()
with open("./datasets/roco/test/cui_embeddings.pkl", "wb") as f:
    pkl.dump(cui_embeddings.cpu().detach(), f)